# Подбор метода уровней и ATR-множителей (буфер / SL / трейлинг) для стратегии отскока

Стратегия bounce: лимитные ордера ставятся **перед** уровнем
- `Buy Limit = S + buffer_atr_s * ATR` (лонг от поддержки S)
- `Sell Limit = R - buffer_atr_r * ATR` (шорт от сопротивления R)

Оптимизируется: метод уровней (Camarilla / Pivot / DeMark) и **шесть** множителей ATR —
буфер, SL и трейлинг **отдельно** для лонга (S) и шорта (R).

**Данные**: положите `EURUSD_H1_2020-01-01_2025-12-31.csv` в папку `content/`
рядом с ноутбуком (локально) или загрузите его в `/content/` в Colab.
Пути подхватятся автоматически.

Запускайте ячейки сверху вниз. Полный подбор занимает ~20 минут
(480 комбинаций на 2020–2024 + проверка на 2025).


In [1]:
# В Colab backtesting обычно уже стоит, но подстрахуемся
try:
    import backtesting
    print("backtesting уже установлен:", backtesting.__version__)
    import matplotlib.pyplot as plt
    print("matplotlib уже установлен:")

except ImportError:
    get_ipython().system('pip install -q backtesting')
    import backtesting
    print("backtesting установлен:", backtesting.__version__)
    get_ipython().system('pip install -q matplotlib')
    import matplotlib.pyplot as plt
    print("matplotlib установлен:")


/workspaces/FV_Bounce/.venv-1/lib/python3.14/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

backtesting уже установлен: 0.6.6
matplotlib уже установлен:


In [2]:
import os
import warnings
from itertools import product

import numpy as np
import pandas as pd

from backtesting import Backtest, Strategy

# import backtesting.backtesting as _bb # Отключаем tqdm-прогресс каждого прогона — при 480+ запусках он забивает лог
# _bb._tqdm = lambda iterable, **kwargs: iterable
# Штатное предупреждение backtesting.py про SL/TP в том же баре, что и вход
warnings.filterwarnings(
    "ignore",
    message=".*contingent SL/TP order would execute in the same bar.*",
)

CONFIG = {
    "data_file": "EURUSD_H1_2020-01-01_2025-12-31.csv",
    "atr_period": 14,
    # Сдвиг начала суток для расчёта дневных уровней.
    # 0 = полночь UTC по времени баров. Если уровни в MT5 считаются
    # по серверным суткам (обычно UTC+2/+3), задайте 2 или 3.
    "day_offset_hours": 0,
    "split_date": "2025-01-01",   # подбор до этой даты, проверка с неё
    "cash": 100_000,
    "leverage": 100,                # кредитное плечо счета 1:100
    "commission": 0.00002,        # ~0.002% (реалистично для 0.01 лота EURUSD)
    "min_trades": 30,             # минимум сделок для отсева дегенерата
    "top_n": 10,                  # сколько комбинаций показывать в топе
    "max_risk_per_trade_pct": 0.02, # Максимальный риск на сделку в % от эквити
    "max_margin_failures": 3,     # остановка подбора после отказов по марже
}

# Сетки множителей (одинаковые для S и R)
BUFFER_GRID = [0.1, 0.15, 0.2, 0.25]
SL_GRID     = [0.5, 1.0, 1.5, 2.0]
TP_GRID     = [1.0, 1.5, 2.0, 2.5, 3.0, 4.0] # Новая сетка для Take Profit

LEVEL_METHOD_NAMES = {0: "Camarilla", 1: "Pivot (классика)", 2: "DeMark"}

# Поддержки/сопротивления по методам (колонки, создаваемые в prepare)
SUP_COLS = {
    0: ["cam_s1", "cam_s2", "cam_s3", "cam_s4"],
    1: ["piv_s1", "piv_s2", "piv_s3"],
    2: ["dem_s1"],
}
RES_COLS = {
    0: ["cam_r1", "cam_r2", "cam_r3", "cam_r4"],
    1: ["piv_r1", "piv_r2", "piv_r3"],
    2: ["dem_r1"],
}

# --- путь к данным: пробуем варианты (Colab: /content/, локально: content/) ---
_CANDIDATE_PATHS = [
    "content/EURUSD_H1_2020-01-01_2025-12-31.csv",          # локально, папка content/
    "/content/EURUSD_H1_2020-01-01_2025-12-31.csv",         # Colab: файл в /content/
    "/content/content/EURUSD_H1_2020-01-01_2025-12-31.csv", # Colab: папка content/ внутри /content/
    "EURUSD_H1_2020-01-01_2025-12-31.csv",                  # текущая папка
]
DATA_PATH = next((p for p in _CANDIDATE_PATHS if os.path.exists(p)), None)
if DATA_PATH is None:
    try:
        from google.colab import files  # только в Colab
        print("CSV не найден. Загрузите файл",
              CONFIG["data_file"], "в /content/ ...")
        uploaded = files.upload()
        DATA_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(
            f"Положите {CONFIG['data_file']} в папку content/ рядом с ноутбуком"
        )
print("Данные:", DATA_PATH)

Данные: content/EURUSD_H1_2020-01-01_2025-12-31.csv


In [3]:
# ---------- Уровни (формулы как в индикаторах MT5) ----------

def camarilla_levels(d1):
    """Camarilla: R1-R4 / S1-S4, коэф. 1.1/12, /6, /4, /2."""
    h, l, c = d1["High"], d1["Low"], d1["Close"]
    hl = h - l
    out = pd.DataFrame(index=d1.index)
    for i, div in enumerate([12, 6, 4, 2], start=1):
        out[f"cam_r{i}"] = c + hl * 1.1 / div
        out[f"cam_s{i}"] = c - hl * 1.1 / div
    return out


def pivot_levels(d1):
    """Классические Pivot: P, R1-R3 / S1-S3."""
    h, l, c = d1["High"], d1["Low"], d1["Close"]
    p = (h + l + c) / 3
    out = pd.DataFrame(index=d1.index)
    out["piv_r1"] = 2 * p - l
    out["piv_s1"] = 2 * p - h
    out["piv_r2"] = p + (h - l)
    out["piv_s2"] = p - (h - l)
    out["piv_r3"] = h + 2 * (p - l)
    out["piv_s3"] = l - 2 * (h - p)
    return out


def demark_levels(d1):
    """DeMark: P, R1 / S1 (X зависит от close vs open)."""
    h, l, c, o = d1["High"], d1["Low"], d1["Close"], d1["Open"]
    x = pd.Series(
        np.where(c < o, h + 2 * l + c,
                 np.where(c > o, 2 * h + l + c, h + l + 2 * c)),
        index=d1.index,
    )
    out = pd.DataFrame(index=d1.index)
    out["dem_r1"] = x / 2 - l
    out["dem_s1"] = x / 2 - h
    return out


In [4]:
def load_and_prepare(csv_path, cfg):
    df = pd.read_csv(csv_path)
    df["time"] = pd.to_datetime(df["time"])
    df = df.set_index("time").sort_index()
    df = df.rename(columns={
        "open": "Open", "high": "High", "low": "Low",
        "close": "Close", "tick_volume": "Volume",
    })[["Open", "High", "Low", "Close", "Volume"]]

    # --- дневные уровни из H1 (граница дня = полночь + day_offset_hours) ---
    off = pd.Timedelta(hours=cfg["day_offset_hours"])
    day = (df.index + off).normalize()
    d1 = df.groupby(day).agg(
        Open=("Open", "first"), High=("High", "max"),
        Low=("Low", "min"), Close=("Close", "last"),
    )
    lev = pd.concat(
        [camarilla_levels(d1), pivot_levels(d1), demark_levels(d1)], axis=1
    )
    lev = lev.shift(1)          # уровни ПРЕДЫДУЩЕГО дня
    lev.index.name = "_day"
    df = df.assign(_day=day).join(lev, on="_day").drop(columns="_day")

    # --- ATR(period) ---
    prev = df["Close"].shift(1)
    tr = pd.concat(
        [df["High"] - df["Low"],
         (df["High"] - prev).abs(),
         (df["Low"] - prev).abs()],
        axis=1,
    ).max(axis=1)
    df["ATR"] = tr.rolling(cfg["atr_period"]).mean()
    return df


In [5]:
class MarginAttemptLimitReached(RuntimeError):
    pass


class BounceStrategy(Strategy):
    level_method = 0            # 0=Camarilla, 1=Pivot, 2=DeMark
    buffer_atr_s = 0.5          # лонг: ордер на S + buffer_atr_s * ATR
    buffer_atr_r = 0.5          # шорт: ордер на R - buffer_atr_r * ATR
    sl_atr_s = 2.0
    sl_atr_r = 2.0
    tp_atr_s = 2.0              # Take Profit для лонга
    tp_atr_r = 2.0              # Take Profit для шорта
    long_only = False
    short_only = False
    max_risk_per_trade_pct = 0.02 # Максимальный риск на сделку в % от эквити (по умолчанию)
    max_margin_failures = 3

    # Ограничения: максимум 3 открытых позиции, размер лота не более 0.1 лота
    # (0.1 лота EURUSD = 10 000 единиц базовой валюты)
    max_open_positions = 3
    max_lot_size = 0.1
    lot_units = 100_000          # 1.0 лот = 100 000 единиц (для EURUSD)
    max_position_size = max_lot_size * lot_units  # = 10 000 единиц

    def init(self):
        # numpy-массивы вместо pandas-обращений — горячий путь next()
        df = self.data.df
        self._i = 0
        self._close = df["Close"].to_numpy()
        self._atr = df["ATR"].to_numpy()
        self._sup = {c: df[c].to_numpy() for c in SUP_COLS[self.level_method]}
        self._res = {c: df[c].to_numpy() for c in RES_COLS[self.level_method]}
        self._margin_failures = 0
        # Собственный счётчик открытых позиций (в backtesting.py 0.6.6
        # доступен только self.position — одна агрегированная позиция,
        # поэтому для лимитных ордеров считаем открытые вручную)
        self._open_positions = 0

    def _nearest_support(self, i, close):
        best = -np.inf
        for arr in self._sup.values():
            v = arr[i]
            if v <= close and v > best:      # NaN проваливает оба сравнения
                best = v
        return best if best > -np.inf else None

    def _nearest_resistance(self, i, close):
        best = np.inf
        for arr in self._res.values():
            v = arr[i]
            if v >= close and v < best:
                best = v
        return best if best < np.inf else None

    def _has_margin(self, size, entry):
        broker = getattr(self, '_broker', None) or getattr(self, 'broker', None)
        margin_available = getattr(broker, 'margin_available', np.inf)
        leverage = getattr(broker, '_leverage', 1.0)
        required_margin = abs(size) * entry / leverage
        if required_margin <= margin_available:
            self._margin_failures = 0
            return True
        self._margin_failures += 1
        print(f"[BounceStrategy.next id={id(self)}] Недостаточная маржа: "
              f"нужно {required_margin:.2f}, доступно {margin_available:.2f}; "
              f"попытка {self._margin_failures}/{self.max_margin_failures}")
        if self._margin_failures >= self.max_margin_failures:
            raise MarginAttemptLimitReached(
                f"достигнут лимит неудачных попыток по марже: {self.max_margin_failures}"
            )
        return False

    def _cap_size(self, size):
        """Ограничиваем размер позиции максимумом max_position_size (0.1 лота)."""
        return min(max(1, int(round(size))), int(self.max_position_size))

    def next(self):
        # Безопасно получаем broker (может быть недоступен в некоторых окружениях)
        broker = getattr(self, '_broker', None) or getattr(self, 'broker', None)
        equity_for_sizing = getattr(broker, 'equity', CONFIG.get('cash', 0))

        i = self._i
        self._i += 1

        # отменяем старые незаполненные ордера (но НЕ contingent SL/TP сделок)
        for o in list(self.orders):
            if not o.is_contingent:
                o.cancel()

        # Синхронизируем собственный счётчик с реальным состоянием:
        # позиция закрыта (SL/TP сработал), если self.position больше не открыт.
        position = getattr(self, 'position', None)
        if position is None or position.size == 0:
            self._open_positions = 0

        # На новом баре выставляем лимитные ордера как обычно, НО позиция
        # открывается только если открытых позиций меньше максимума (3).
        if self._open_positions >= self.max_open_positions:
            return

        atr = self._atr[i]
        if not np.isfinite(atr) or atr <= 0:
            return
        close = self._close[i]

        # --- лимитные ордера перед уровнем (bounce) ---
        if not self.short_only:
            sup = self._nearest_support(i, close)
            if sup is not None:
                entry = sup + self.buffer_atr_s * atr
                risk_per_unit_long = self.sl_atr_s * atr
                if risk_per_unit_long <= 0: # Пропускаем, если риск некорректен
                    print(f"[BounceStrategy.next id={id(self)}] Предупреждение: Некорректный 'risk_per_unit_long'. Пропускаем лонг.")
                    return
                trade_size_long = (equity_for_sizing * self.max_risk_per_trade_pct) / risk_per_unit_long
                trade_size_long = self._cap_size(trade_size_long)
                if close > entry and self._has_margin(trade_size_long, entry):
                    self.buy(size=trade_size_long, limit=entry,
                             sl=entry - self.sl_atr_s * atr,
                             tp=entry + self.tp_atr_s * atr)
                    self._open_positions += 1

        if not self.long_only:
            res = self._nearest_resistance(i, close)
            if res is not None:
                entry = res - self.buffer_atr_r * atr
                risk_per_unit_short = self.sl_atr_r * atr
                if risk_per_unit_short <= 0: # Пропускаем, если риск некорректен
                    print(f"[BounceStrategy.next id={id(self)}] Предупреждение: Некорректный 'risk_per_unit_short'. Пропускаем шорт.")
                    return
                trade_size_short = (equity_for_sizing * self.max_risk_per_trade_pct) / risk_per_unit_short
                trade_size_short = self._cap_size(trade_size_short)
                if close < entry and self._has_margin(trade_size_short, entry):
                    self.sell(size=trade_size_short, limit=entry,
                              sl=entry + self.sl_atr_r * atr,
                              tp=entry - self.tp_atr_r * atr)
                    self._open_positions += 1

In [6]:
 # ---------- Помощники подбора ----------

def stats_row(s, params):
    return {
        **params,
        "Sharpe": s.get("Sharpe Ratio", np.nan),
        "Return %": s.get("Return [%]", np.nan),
        "SQN": s.get("SQN", np.nan),
        "WinRate %": s.get("Win Rate [%]", np.nan),
        "Trades": s.get("# Trades", len(s.get("_trades", []))),
        "DD %": s.get("Max. Drawdown [%]", np.nan),
        "Profit Factor": s.get("Profit Factor", np.nan),
        "End Balance": s.get("End Balance", np.nan),
    }


def grid_search(bt_obj, grids, fixed, cfg):
    keys = list(grids)
    combos = list(product(*grids.values()))
    rows = []
    skipped_sl_tp = 0
    BounceStrategy.max_margin_failures = cfg.get("max_margin_failures", 3)
    for k, combo in enumerate(combos, 1):
        params = dict(zip(keys, combo))
        # run() принимает скаляры; в fixed значения лежат списками
        params.update({k: v[0] for k, v in fixed.items()})

        # Раньше фильтр исключал комбинации с SL <= TP — это неверно.
        # Теперь пропускаем только те комбинации, где SL > TP (неграмотно задано TP).
        invalid_s = not params["short_only"] and params["sl_atr_s"] > params["tp_atr_s"]
        invalid_r = not params["long_only"] and params["sl_atr_r"] > params["tp_atr_r"]
        if invalid_s or invalid_r:
            skipped_sl_tp += 1
            sides = []
            if invalid_s:
                sides.append(f"long SL={params['sl_atr_s']} > TP={params['tp_atr_s']}")
            if invalid_r:
                sides.append(f"short SL={params['sl_atr_r']} > TP={params['tp_atr_r']}")
            # print(f"    пропускается комбинация {k}/{len(combos)}: " + "; ".join(sides), flush=True)
            continue

        try:
            s = bt_obj.run(**params)
        except MarginAttemptLimitReached as exc:
            print(f"    подбор остановлен на комбинации {k}/{len(combos)}: {exc}", flush=True)
            break
        rows.append(stats_row(s, params))
        if k % 50 == 0 or k == len(combos):
            print(f"    прогресс {k}/{len(combos)}", flush=True)

    print(f"    пропущено комбинаций SL > TP: {skipped_sl_tp}/{len(combos)}", flush=True)
    if not rows:
        raise ValueError("После фильтра SL > TP не осталось допустимых комбинаций")
    out = pd.DataFrame(rows)
    out = out[out["Trades"] >= cfg["min_trades"]]
    if out.empty:
        out = pd.DataFrame(rows)          # если все комбинации малодеятельные
    return out.sort_values("Sharpe", ascending=False).reset_index(drop=True)


def top_table(rows, cfg):
    d = rows.head(cfg["top_n"]).copy()
    d["Метод"] = d["level_method"].map(LEVEL_METHOD_NAMES)
    # Обновляем названия колонок с trailing на tp
    cols = ["Метод",
            "buffer_atr_s", "sl_atr_s", "tp_atr_s",
            "buffer_atr_r", "sl_atr_r", "tp_atr_r",
            "Sharpe", "Return %", "SQN", "WinRate %", "Trades", "DD %",
            "Profit Factor", "End Balance"]
    return d[cols]


In [7]:
class BreakoutStrategy(BounceStrategy):
    """Breakout-версия: вход stop за ближайшим сопротивлением/поддержкой."""

    def next(self):
        broker = getattr(self, "_broker", None) or getattr(self, "broker", None)
        equity_for_sizing = getattr(broker, "equity", CONFIG.get("cash", 0))

        i = self._i
        self._i += 1

        for order in list(self.orders):
            if not order.is_contingent:
                order.cancel()

        position = getattr(self, "position", None)
        if position is None or position.size == 0:
            self._open_positions = 0
        if self._open_positions >= self.max_open_positions:
            return

        atr = self._atr[i]
        close = self._close[i]
        if not np.isfinite(atr) or atr <= 0 or not np.isfinite(close):
            return

        if not self.short_only:
            resistance = self._nearest_resistance(i, close)
            if resistance is not None:
                entry = resistance + self.buffer_atr_s * atr
                risk = self.sl_atr_s * atr
                if risk > 0:
                    size = self._cap_size((equity_for_sizing * self.max_risk_per_trade_pct) / risk)
                    if entry > close and self._has_margin(size, entry):
                        self.buy(
                            size=size,
                            stop=entry,
                            sl=entry - self.sl_atr_s * atr,
                            tp=entry + self.tp_atr_s * atr,
                        )
                        self._open_positions += 1

        if self._open_positions >= self.max_open_positions or self.long_only:
            return

        support = self._nearest_support(i, close)
        if support is not None:
            entry = support - self.buffer_atr_r * atr
            risk = self.sl_atr_r * atr
            if risk > 0:
                size = self._cap_size((equity_for_sizing * self.max_risk_per_trade_pct) / risk)
                if entry < close and self._has_margin(size, entry):
                    self.sell(
                        size=size,
                        stop=entry,
                        sl=entry + self.sl_atr_r * atr,
                        tp=entry - self.tp_atr_r * atr,
                    )
                    self._open_positions += 1

In [8]:
df = load_and_prepare(DATA_PATH, CONFIG)
# Уменьшаем выборку до 2024-2025 годов, как запрошено пользователем
df = df.loc['2024':'2025']
print(f"Баров: {len(df)}, период {df.index[0]} .. {df.index[-1]}")

split = pd.Timestamp(CONFIG["split_date"])
# df_train будет содержать данные до split_date (т.е. 2024 год)
df_train = df[df.index < split]
# df_test будет содержать данные с split_date (т.е. 2025 год)
df_test = df[df.index >= split]

print(f"Подбор: {len(df_train)} баров ({df_train.index[0].date()} .. {df_train.index[-1].date()}), "
      f"проверка: {len(df_test)} баров ({df_test.index[0].date()} .. {df_test.index[-1].date()})")

bt_train = Backtest(df_train, BounceStrategy,
                    cash=CONFIG["cash"], commission=CONFIG["commission"],
                    margin=1 / CONFIG["leverage"],
                    finalize_trades=True)
bt_test = Backtest(df_test, BounceStrategy,
                   cash=CONFIG["cash"], commission=CONFIG["commission"],
                   margin=1 / CONFIG["leverage"],
                   finalize_trades=True)

Баров: 12396, период 2024-01-02 00:00:00 .. 2025-12-30 21:00:00
Подбор: 6226 баров (2024-01-02 .. 2024-12-31), проверка: 6170 баров (2025-01-02 .. 2025-12-30)


In [9]:
# Проход: ОТСКОК ОТ ПОДДЕРЖКИ — тестируется: Buy Limit
print("Проход: ОТСКОК ОТ ПОДДЕРЖКИ — подбор S-множителей (Buy Limit)")
r1 = grid_search(
    bt_train,
    grids={
        "level_method": range(3),
        "buffer_atr_s": BUFFER_GRID,
        "sl_atr_s": SL_GRID,
        "tp_atr_s": TP_GRID,
    },
    fixed={
        "buffer_atr_r": [0.5], "sl_atr_r": [1.0], "tp_atr_r": [2.0],
        "long_only": [True], "short_only": [False],
    },
    cfg=CONFIG,
)
best1 = r1.iloc[0]
display(top_table(r1, CONFIG))

Проход: ОТСКОК ОТ ПОДДЕРЖКИ — подбор S-множителей (Buy Limit)
    прогресс 50/288
    прогресс 100/288
    прогресс 150/288
    прогресс 200/288
    прогресс 250/288
    прогресс 288/288
    пропущено комбинаций SL > TP: 36/288


,Метод,buffer_atr_s,sl_atr_s,tp_atr_s,buffer_atr_r,sl_atr_r,tp_atr_r,Sharpe,Return %,SQN,WinRate %,Trades,DD %,Profit Factor,End Balance
0,Pivot (классика),0.15,1.5,4.0,0.5,1.0,2.0,1.729548,1.181962,2.188835,33.227848,316,-0.671472,1.367865,NaN
1,Pivot (классика),0.20,1.5,4.0,0.5,1.0,2.0,1.706041,1.169193,2.117280,33.757962,314,-0.699287,1.356135,NaN
2,Pivot (классика),0.25,2.0,4.0,0.5,1.0,2.0,1.573447,1.098318,1.930623,37.676056,284,-0.886030,1.316013,NaN
3,Pivot (классика),0.25,1.5,4.0,0.5,1.0,2.0,1.568606,1.043294,1.890324,33.121019,314,-0.753748,1.312223,NaN
4,Pivot (классика),0.20,2.0,4.0,0.5,1.0,2.0,1.481878,1.005876,1.788936,36.879433,282,-0.820979,1.293607,NaN
5,Pivot (классика),0.20,1.5,2.0,0.5,1.0,2.0,1.476982,0.740866,1.736870,45.528455,369,-0.453771,1.242024,NaN
6,Pivot (классика),0.20,1.5,2.5,0.5,1.0,2.0,1.471425,0.792493,1.718423,40.909091,352,-0.485463,1.247707,NaN
7,Pivot (классика),0.15,1.5,2.5,0.5,1.0,2.0,1.438883,0.768707,1.735289,40.462428,346,-0.516627,1.251490,NaN
8,Pivot (классика),0.15,1.5,2.0,0.5,1.0,2.0,1.383780,0.692720,1.643006,44.959128,367,-0.423221,1.228598,NaN
9,DeMark,0.15,2.0,4.0,0.5,1.0,2.0,1.360020,0.882150,1.670934,39.130435,253,-0.598885,1.299822,NaN


In [10]:
# ---------- Breakout: подготовка Backtest (запустите следующие ячейки отдельно) ----------
brk_train = Backtest(df_train, BreakoutStrategy,
                     cash=CONFIG["cash"], commission=CONFIG["commission"],
                     margin=1 / CONFIG["leverage"],
                     finalize_trades=True)
brk_test = Backtest(df_test, BreakoutStrategy,
                    cash=CONFIG["cash"], commission=CONFIG["commission"],
                    margin=1 / CONFIG["leverage"],
                    finalize_trades=True)
print("Breakout Backtest подготовлен. Запустите ячейки 'Breakout LONG' и 'Breakout SELL'.")

Breakout Backtest подготовлен. Запустите ячейки 'Breakout LONG' и 'Breakout SELL'.


In [11]:
# ПРОБОЙ СОПРОТИВЛЕНИЯ — ПРОХОД: ПРОБОЙ СОПРОТИВЛЕНИЯ (BUY STOP)
print("ПРОБОЙ СОПРОТИВЛЕНИЯ — подбор: level_method, buffer_s, sl_s, tp_s (Buy Stop)")
brk_r1 = grid_search(
    brk_train,
    grids={
        "level_method": range(3),
        "buffer_atr_s": BUFFER_GRID,
        "sl_atr_s": SL_GRID,
        "tp_atr_s": TP_GRID,
    },
    fixed={
        "buffer_atr_r": [0.5], "sl_atr_r": [1.0], "tp_atr_r": [2.0],
        "long_only": [True], "short_only": [False],
    },
    cfg=CONFIG,
)
brk_best1 = brk_r1.iloc[0]
display(top_table(brk_r1, CONFIG))

ПРОБОЙ СОПРОТИВЛЕНИЯ — подбор: level_method, buffer_s, sl_s, tp_s (Buy Stop)
    прогресс 50/288
    прогресс 100/288
    прогресс 150/288
    прогресс 200/288
    прогресс 250/288
    прогресс 288/288
    пропущено комбинаций SL > TP: 36/288


,Метод,buffer_atr_s,sl_atr_s,tp_atr_s,buffer_atr_r,sl_atr_r,tp_atr_r,Sharpe,Return %,SQN,WinRate %,Trades,DD %,Profit Factor,End Balance
0,Camarilla,0.25,2.0,2.0,0.5,1.0,2.0,-0.877665,-0.626269,-1.248015,50.511945,586,-0.849858,0.893087,NaN
1,Camarilla,0.20,2.0,2.0,0.5,1.0,2.0,-0.882020,-0.647315,-1.238229,50.495050,606,-0.800235,0.894554,NaN
2,Camarilla,0.25,2.0,2.5,0.5,1.0,2.0,-0.975909,-0.784440,-1.463255,44.990177,509,-1.028333,0.867985,NaN
3,Camarilla,0.20,2.0,3.0,0.5,1.0,2.0,-1.008249,-0.882565,-1.568557,41.176471,459,-1.177449,0.850974,NaN
4,Camarilla,0.25,2.0,3.0,0.5,1.0,2.0,-1.047390,-0.873915,-1.598053,41.071429,448,-1.158569,0.847006,NaN
5,Camarilla,0.25,2.0,4.0,0.5,1.0,2.0,-1.054263,-0.974185,-1.666844,34.196891,386,-1.149554,0.828266,NaN
6,Camarilla,0.20,2.0,2.5,0.5,1.0,2.0,-1.089456,-0.916797,-1.647604,45.142857,525,-1.155891,0.854209,NaN
7,Camarilla,0.20,1.5,2.0,0.5,1.0,2.0,-1.135738,-0.763639,-1.577025,45.321637,684,-0.967239,0.876079,NaN
8,Camarilla,0.10,2.0,2.5,0.5,1.0,2.0,-1.178552,-1.074064,-1.831668,44.547135,541,-1.210577,0.842854,NaN
9,Camarilla,0.10,2.0,3.0,0.5,1.0,2.0,-1.195304,-1.153522,-1.933443,39.583333,480,-1.321306,0.823210,NaN


In [12]:
# ПРОБОЙ ПОДДЕРЖКИ — ПРОХОД: ПРОБОЙ ПОДДЕРЖКИ (SELL STOP)
print("ПРОБОЙ ПОДДЕРЖКИ — подбор: level_method, buffer_r, sl_r, tp_r (Sell Stop)")
brk_r2 = grid_search(
    brk_train,
    grids={
        "level_method": range(3),
        "buffer_atr_r": BUFFER_GRID,
        "sl_atr_r": SL_GRID,
        "tp_atr_r": TP_GRID,
    },
    fixed={
        "buffer_atr_s": [0.5], "sl_atr_s": [1.0], "tp_atr_s": [2.0],
        "long_only": [False], "short_only": [True],
    },
    cfg=CONFIG,
)
brk_best2 = brk_r2.iloc[0]
display(top_table(brk_r2, CONFIG))

ПРОБОЙ ПОДДЕРЖКИ — подбор: level_method, buffer_r, sl_r, tp_r (Sell Stop)
    прогресс 50/288
    прогресс 100/288
    прогресс 150/288
    прогресс 200/288
    прогресс 250/288
    прогресс 288/288
    пропущено комбинаций SL > TP: 36/288


,Метод,buffer_atr_s,sl_atr_s,tp_atr_s,buffer_atr_r,sl_atr_r,tp_atr_r,Sharpe,Return %,SQN,WinRate %,Trades,DD %,Profit Factor,End Balance
0,Pivot (классика),0.5,1.0,2.0,0.25,1.0,2.5,0.500160,0.272866,0.812758,37.373737,297,-0.291320,1.116933,NaN
1,Pivot (классика),0.5,1.0,2.0,0.25,1.0,4.0,0.398702,0.308485,0.769442,28.015564,257,-0.301807,1.122147,NaN
2,Pivot (классика),0.5,1.0,2.0,0.15,0.5,2.5,0.342672,0.179002,0.587063,29.310345,348,-0.348461,1.085065,NaN
3,Pivot (классика),0.5,1.0,2.0,0.25,1.0,2.0,0.316337,0.168551,0.539626,42.258065,310,-0.250604,1.073563,NaN
4,Pivot (классика),0.5,1.0,2.0,0.15,0.5,2.0,0.312015,0.147650,0.522967,34.173669,357,-0.326058,1.073954,NaN
5,Pivot (классика),0.5,1.0,2.0,0.20,1.0,2.5,0.306996,0.184665,0.511627,37.500000,312,-0.294797,1.069453,NaN
6,Pivot (классика),0.5,1.0,2.0,0.10,0.5,2.5,0.299688,0.155743,0.497247,29.085873,361,-0.246032,1.069000,NaN
7,DeMark,0.5,1.0,2.0,0.20,0.5,2.0,0.269606,0.130071,0.501839,32.394366,284,-0.333142,1.076086,NaN
8,DeMark,0.5,1.0,2.0,0.25,0.5,2.0,0.255569,0.126902,0.490195,32.720588,272,-0.365132,1.074525,NaN
9,Pivot (классика),0.5,1.0,2.0,0.20,1.0,4.0,0.198979,0.196336,0.485496,27.238806,268,-0.342066,1.070784,NaN


In [13]:
# Проход: ОТСКОК ОТ СОПРОТИВЛЕНИЯ — тестируется: Sell Limit
print("Проход: ОТСКОК ОТ СОПРОТИВЛЕНИЯ — подбор: level_method, buffer_r, sl_r, tp_r (Sell Limit)")
r2 = grid_search(
    bt_train,
    grids={
        "level_method": range(3),
        "buffer_atr_r": BUFFER_GRID,
        "sl_atr_r": SL_GRID,
        "tp_atr_r": TP_GRID, # Используем TP_GRID вместо TRAIL_GRID
    },
    fixed={
        "buffer_atr_s": [0.5], "sl_atr_s": [1.0], "tp_atr_s": [2.0], # Устанавливаем TP_atr_s по умолчанию
        "long_only": [False], "short_only": [True],
    },
    cfg=CONFIG,
)
best2 = r2.iloc[0]
display(top_table(r2, CONFIG))


Проход: ОТСКОК ОТ СОПРОТИВЛЕНИЯ — подбор: level_method, buffer_r, sl_r, tp_r (Sell Limit)
    прогресс 50/288
    прогресс 100/288
    прогресс 150/288
    прогресс 200/288
    прогресс 250/288
    прогресс 288/288
    пропущено комбинаций SL > TP: 36/288


,Метод,buffer_atr_s,sl_atr_s,tp_atr_s,buffer_atr_r,sl_atr_r,tp_atr_r,Sharpe,Return %,SQN,WinRate %,Trades,DD %,Profit Factor,End Balance
0,DeMark,0.5,1.0,2.0,0.10,1.5,2.5,2.147412,1.143473,3.075617,43.478261,299,-0.272765,1.509832,NaN
1,DeMark,0.5,1.0,2.0,0.10,1.5,2.0,2.142123,1.044990,3.054173,47.619048,315,-0.291445,1.479438,NaN
2,DeMark,0.5,1.0,2.0,0.15,1.5,2.0,2.092580,1.068197,3.097182,47.896440,309,-0.294776,1.503507,NaN
3,DeMark,0.5,1.0,2.0,0.20,1.5,1.5,2.075031,0.978361,2.995188,52.124646,353,-0.307811,1.442967,NaN
4,DeMark,0.5,1.0,2.0,0.15,1.5,1.5,2.060751,0.933096,2.962398,53.216374,342,-0.270652,1.444472,NaN
5,DeMark,0.5,1.0,2.0,0.10,1.0,2.0,2.022397,0.801931,2.556117,39.701493,335,-0.265867,1.405658,NaN
6,DeMark,0.5,1.0,2.0,0.15,1.5,2.5,2.010216,1.103252,2.972492,44.097222,288,-0.309219,1.505081,NaN
7,DeMark,0.5,1.0,2.0,0.20,1.5,2.0,2.000895,1.043693,2.917701,46.540881,318,-0.316338,1.461624,NaN
8,Camarilla,0.5,1.0,2.0,0.15,1.5,2.0,1.959807,1.641848,2.804245,43.883985,793,-0.315324,1.253572,NaN
9,Pivot (классика),0.5,1.0,2.0,0.25,2.0,2.0,1.898613,1.159514,2.696135,50.696379,359,-0.378298,1.367581,NaN


In [18]:
# --- Проход 3: выбор метода отдельно для всех 4 стратегий ---

# Проверки: убедимся, что предыдущие проходы выполнены
if 'best1' not in globals():
    raise NameError("Переменная 'best1' не определена. Запустите ячейку 'Проход 1' (подбор LONG для Bounce) перед этой ячейкой.")
if 'best2' not in globals():
    raise NameError("Переменная 'best2' не определена. Запустите ячейку 'Проход 2' (подбор SHORT для Bounce) перед этой ячейкой.")
if 'brk_best1' not in globals():
    raise NameError("Переменная 'brk_best1' не определена. Запустите ячейку 'Breakout LONG' перед этой ячейкой.")
if 'brk_best2' not in globals():
    raise NameError("Переменная 'brk_best2' не определена. Запустите ячейку 'Breakout SELL' перед этой ячейкой.")
if 'bt_train' not in globals() or 'brk_train' not in globals():
    raise NameError("Backtest объекты bt_train/brk_train не найдены. Запустите подготовительные ячейки с Backtest перед этой ячейкой.")
# Breakout: метод для ЛОНГ (по лучшим S-множителям из brk_best1)
print("ПРОБОЙ СОПРОТИВЛЕНИЯ — ПРОХОД: ПРОБОЙ СОПРОТИВЛЕНИЯ (BUY STOP) — выбор метода для ЛОНГ при лучших S-множителях")
brk_r3_long = grid_search(
    brk_train,
    grids={"level_method": range(3)},
    fixed={
        "buffer_atr_s": [brk_best1["buffer_atr_s"]],
        "sl_atr_s": [brk_best1["sl_atr_s"]],
        "tp_atr_s": [brk_best1["tp_atr_s"]],
        "buffer_atr_r": [0.5], "sl_atr_r": [1.0], "tp_atr_r": [2.0],
        "long_only": [True], "short_only": [False],
    },
    cfg=CONFIG,
)
brk_best_long = brk_r3_long.iloc[0]
display(top_table(brk_r3_long, CONFIG))

# Breakout: метод для ШОРТ (по лучшим R-множителям из brk_best2)
print("ПРОБОЙ ПОДДЕРЖКИ — ПРОХОД: ПРОБОЙ ПОДДЕРЖКИ (SELL STOP) — выбор метода для ШОРТ при лучших R-множителях")
brk_r3_short = grid_search(
    brk_train,
    grids={"level_method": range(3)},
    fixed={
        "buffer_atr_s": [0.5], "sl_atr_s": [1.0], "tp_atr_s": [2.0],
        "buffer_atr_r": [brk_best2["buffer_atr_r"]],
        "sl_atr_r": [brk_best2["sl_atr_r"]],
        "tp_atr_r": [brk_best2["tp_atr_r"]],
        "long_only": [False], "short_only": [True],
    },
    cfg=CONFIG,
)
brk_best_short = brk_r3_short.iloc[0]
display(top_table(brk_r3_short, CONFIG))

# Bounce: метод для ЛОНГ (по лучшим S-множителям из best1)
print("ОТСКОК ОТ ПОДДЕРЖКИ — ПРОХОД: ОТСКОК ОТ ПОДДЕРЖКИ (Buy Limit) — выбор метода для ЛОНГ при лучших S-множителях")
r3_long = grid_search(
    bt_train,
    grids={"level_method": range(3)},
    fixed={
        "buffer_atr_s": [best1["buffer_atr_s"]],
        "sl_atr_s": [best1["sl_atr_s"]],
        "tp_atr_s": [best1.get("tp_atr_s", 2.0)],
        "buffer_atr_r": [0.5], "sl_atr_r": [1.0], "tp_atr_r": [2.0],
        "long_only": [True], "short_only": [False],
    },
    cfg=CONFIG,
)
best_long = r3_long.iloc[0]
display(top_table(r3_long, CONFIG))

# Bounce: метод для ШОРТ (по лучшим R-множителям из best2)
print("ОТСКОК ОТ СОПРОТИВЛЕНИЯ — ПРОХОД: ОТСКОК ОТ СОПРОТИВЛЕНИЯ (Sell Limit) — выбор метода для ШОРТ при лучших R-множителях")
r3_short = grid_search(
    bt_train,
    grids={"level_method": range(3)},
    fixed={
        "buffer_atr_s": [0.5], "sl_atr_s": [1.0], "tp_atr_s": [2.0],
        "buffer_atr_r": [best2["buffer_atr_r"]],
        "sl_atr_r": [best2["sl_atr_r"]],
        "tp_atr_r": [best2.get("tp_atr_r", 2.0)],
        "long_only": [False], "short_only": [True],
    },
    cfg=CONFIG,
)
best_short = r3_short.iloc[0]
display(top_table(r3_short, CONFIG))

# Сформируем параметры для проверки на тестовом периоде (Bounce) — эти же переменные использует последующая ячейка
test_params_long = {
    "level_method": int(best_long["level_method"]),
    "buffer_atr_s": float(best_long["buffer_atr_s"]),
    "sl_atr_s": float(best_long["sl_atr_s"]),
    "tp_atr_s": float(best_long.get("tp_atr_s", 2.0)),
    "buffer_atr_r": 0.5, "sl_atr_r": 1.0, "tp_atr_r": 2.0,
    "long_only": True, "short_only": False,
}
test_params_short = {
    "level_method": int(best_short["level_method"]),
    "buffer_atr_s": 0.5, "sl_atr_s": 1.0, "tp_atr_s": 2.0,
    "buffer_atr_r": float(best_short["buffer_atr_r"]),
    "sl_atr_r": float(best_short["sl_atr_r"]),
    "tp_atr_r": float(best_short.get("tp_atr_r", 2.0)),
    "long_only": False, "short_only": True,
}

# Итог: лучшие методы для 4 стратегий
print('\n--- Лучшие методы (Breakout/Bounce) для LONG и SHORT ---')
print('ПРОБОЙ СОПРОТИВЛЕНИЯ — ПРОХОД: ПРОБОЙ СОПРОТИВЛЕНИЯ (BUY STOP) :', LEVEL_METHOD_NAMES[int(brk_best_long['level_method'])], ' — params S:', {
    'buffer_atr_s': float(brk_best1['buffer_atr_s']), 'sl_atr_s': float(brk_best1['sl_atr_s']), 'tp_atr_s': float(brk_best1.get('tp_atr_s', 2.0))
})
print('ПРОБОЙ ПОДДЕРЖКИ — ПРОХОД: ПРОБОЙ ПОДДЕРЖКИ (SELL STOP) :', LEVEL_METHOD_NAMES[int(brk_best_short['level_method'])], ' — params R:', {
    'buffer_atr_r': float(brk_best2['buffer_atr_r']), 'sl_atr_r': float(brk_best2['sl_atr_r']), 'tp_atr_r': float(brk_best2.get('tp_atr_r', 2.0))
})
print('ОТСКОК ОТ ПОДДЕРЖКИ — ПРОХОД: ОТСКОК ОТ ПОДДЕРЖКИ (Buy Limit) :', LEVEL_METHOD_NAMES[int(best_long['level_method'])], ' — params S:', {
    'buffer_atr_s': float(best_long['buffer_atr_s']), 'sl_atr_s': float(best_long['sl_atr_s']), 'tp_atr_s': float(best_long.get('tp_atr_s', 2.0))
})
print('ОТСКОК ОТ СОПРОТИВЛЕНИЯ — ПРОХОД: ОТСКОК ОТ СОПРОТИВЛЕНИЯ (Sell Limit) :', LEVEL_METHOD_NAMES[int(best_short['level_method'])], ' — params R:', {
    'buffer_atr_r': float(best_short['buffer_atr_r']), 'sl_atr_r': float(best_short['sl_atr_r']), 'tp_atr_r': float(best_short.get('tp_atr_r', 2.0))
})

ПРОБОЙ СОПРОТИВЛЕНИЯ — ПРОХОД: ПРОБОЙ СОПРОТИВЛЕНИЯ (BUY STOP) — выбор метода для ЛОНГ при лучших S-множителях
    прогресс 3/3
    пропущено комбинаций SL > TP: 0/3


,Метод,buffer_atr_s,sl_atr_s,tp_atr_s,buffer_atr_r,sl_atr_r,tp_atr_r,Sharpe,Return %,SQN,WinRate %,Trades,DD %,Profit Factor,End Balance
0,Camarilla,0.25,2.0,2.0,0.5,1.0,2.0,-0.877665,-0.626269,-1.248015,50.511945,586,-0.849858,0.893087,NaN
1,Pivot (классика),0.25,2.0,2.0,0.5,1.0,2.0,-1.649429,-0.755386,-2.222571,47.876448,259,-0.854192,0.735047,NaN
2,DeMark,0.25,2.0,2.0,0.5,1.0,2.0,-2.080528,-1.063889,-3.194269,45.038168,262,-1.134030,0.645382,NaN


ПРОБОЙ ПОДДЕРЖКИ — ПРОХОД: ПРОБОЙ ПОДДЕРЖКИ (SELL STOP) — выбор метода для ШОРТ при лучших R-множителях
    прогресс 3/3
    пропущено комбинаций SL > TP: 0/3


,Метод,buffer_atr_s,sl_atr_s,tp_atr_s,buffer_atr_r,sl_atr_r,tp_atr_r,Sharpe,Return %,SQN,WinRate %,Trades,DD %,Profit Factor,End Balance
0,Pivot (классика),0.5,1.0,2.0,0.25,1.0,2.5,0.500160,0.272866,0.812758,37.373737,297,-0.291320,1.116933,NaN
1,DeMark,0.5,1.0,2.0,0.25,1.0,2.5,-1.230176,-0.391273,-1.267297,31.716418,268,-0.673366,0.822935,NaN
2,Camarilla,0.5,1.0,2.0,0.25,1.0,2.5,-1.243370,-0.696211,-1.406112,32.814710,707,-1.030178,0.877722,NaN


ОТСКОК ОТ ПОДДЕРЖКИ — ПРОХОД: ОТСКОК ОТ ПОДДЕРЖКИ (Buy Limit) — выбор метода для ЛОНГ при лучших S-множителях
    прогресс 3/3
    пропущено комбинаций SL > TP: 0/3


,Метод,buffer_atr_s,sl_atr_s,tp_atr_s,buffer_atr_r,sl_atr_r,tp_atr_r,Sharpe,Return %,SQN,WinRate %,Trades,DD %,Profit Factor,End Balance
0,Pivot (классика),0.15,1.5,4.0,0.5,1.0,2.0,1.729548,1.181962,2.188835,33.227848,316,-0.671472,1.367865,NaN
1,DeMark,0.15,1.5,4.0,0.5,1.0,2.0,0.962656,0.544438,1.165545,33.458647,266,-0.557881,1.210635,NaN
2,Camarilla,0.15,1.5,4.0,0.5,1.0,2.0,-0.556867,-0.577788,-0.954429,26.078431,510,-0.925758,0.903001,NaN


ОТСКОК ОТ СОПРОТИВЛЕНИЯ — ПРОХОД: ОТСКОК ОТ СОПРОТИВЛЕНИЯ (Sell Limit) — выбор метода для ШОРТ при лучших R-множителях
    прогресс 3/3
    пропущено комбинаций SL > TP: 0/3


,Метод,buffer_atr_s,sl_atr_s,tp_atr_s,buffer_atr_r,sl_atr_r,tp_atr_r,Sharpe,Return %,SQN,WinRate %,Trades,DD %,Profit Factor,End Balance
0,DeMark,0.5,1.0,2.0,0.1,1.5,2.5,2.147412,1.143473,3.075617,43.478261,299,-0.272765,1.509832,NaN
1,Camarilla,0.5,1.0,2.0,0.1,1.5,2.5,1.102345,0.993044,1.631933,37.170596,721,-0.748075,1.151420,NaN
2,Pivot (классика),0.5,1.0,2.0,0.1,1.5,2.5,1.041939,0.595634,1.424952,37.752161,347,-0.606202,1.191928,NaN



--- Лучшие методы (Breakout/Bounce) для LONG и SHORT ---
ПРОБОЙ СОПРОТИВЛЕНИЯ — ПРОХОД: ПРОБОЙ СОПРОТИВЛЕНИЯ (BUY STOP) : Camarilla  — params S: {'buffer_atr_s': 0.25, 'sl_atr_s': 2.0, 'tp_atr_s': 2.0}
ПРОБОЙ ПОДДЕРЖКИ — ПРОХОД: ПРОБОЙ ПОДДЕРЖКИ (SELL STOP) : Pivot (классика)  — params R: {'buffer_atr_r': 0.25, 'sl_atr_r': 1.0, 'tp_atr_r': 2.5}
ОТСКОК ОТ ПОДДЕРЖКИ — ПРОХОД: ОТСКОК ОТ ПОДДЕРЖКИ (Buy Limit) : Pivot (классика)  — params S: {'buffer_atr_s': 0.15, 'sl_atr_s': 1.5, 'tp_atr_s': 4.0}
ОТСКОК ОТ СОПРОТИВЛЕНИЯ — ПРОХОД: ОТСКОК ОТ СОПРОТИВЛЕНИЯ (Sell Limit) : DeMark  — params R: {'buffer_atr_r': 0.1, 'sl_atr_r': 1.5, 'tp_atr_r': 2.5}


In [16]:
# ---------- ИТОГОВАЯ ТАБЛИЦА: МЕТОД + МНОЖИТЕЛИ + ГДЕ СТАВИТЬ ОРДЕРА ----------
# Лучший метод для ЛОНГ (S-множители из best1, метод из best_long)
m_long = int(test_params_long["level_method"])
# Лучший метод для ШОРТ (R-множители из best2, метод из best_short)
m_short = int(test_params_short["level_method"])

last = df.iloc[-1]
atr, close = last["ATR"], last["Close"]
sup = sup_col = res = res_col = None
for col in SUP_COLS[m_long]:
    v = last[col]
    if np.isfinite(v) and v <= close and (sup is None or v > sup):
        sup, sup_col = v, col
for col in RES_COLS[m_short]:
    v = last[col]
    if np.isfinite(v) and v >= close and (res is None or v < res):
        res, res_col = v, col

b_s = test_params_long["buffer_atr_s"]
s_s = test_params_long["sl_atr_s"]
t_s = test_params_long.get("tp_atr_s", 2.0)
b_r = test_params_short["buffer_atr_r"]
s_r = test_params_short["sl_atr_r"]
t_r = test_params_short.get("tp_atr_r", 2.0)

print("Лучший метод для ЛОНГ :", LEVEL_METHOD_NAMES[m_long])
print(f"  Буфер S : {b_s:.2f} x ATR | SL S: {s_s:.2f} x ATR | TP S: {t_s:.2f} x ATR")
print("Лучший метод для ШОРТ :", LEVEL_METHOD_NAMES[m_short])
print(f"  Буфер R : {b_r:.2f} x ATR | SL R: {s_r:.2f} x ATR | TP R: {t_r:.2f} x ATR")
print("\nПравило постановки ордеров (уровни пересчитываются каждый день):")
print(f"  Buy  Limit = S + {b_s:.2f} * ATR   (лонг от поддержки S, метод {LEVEL_METHOD_NAMES[m_long]})")
print(f"  Sell Limit = R - {b_r:.2f} * ATR   (шорт от сопротивления R, метод {LEVEL_METHOD_NAMES[m_short]})")
if sup is not None and res is not None:
    print(f"\nПример на последнем баре ({last.name}):")
    print(f"  ATR(14) = {atr:.5f}, Close = {close:.5f}")
    print(f"  S = {sup:.5f} ({sup_col})  -> Buy  Limit = {sup + b_s * atr:.5f}")
    print(f"  R = {res:.5f} ({res_col})  -> Sell Limit = {res - b_r * atr:.5f}")

try:
    # Сохраняем результаты отдельно для лонга и шорта
    combined = pd.concat([
        top_table(r1.rename(columns={'tp_atr_s': 'tp_atr_s', 'tp_atr_r': 'tp_atr_r'}), CONFIG).assign(Pass="long"),
        top_table(r2.rename(columns={'tp_atr_s': 'tp_atr_s', 'tp_atr_r': 'tp_atr_r'}), CONFIG).assign(Pass="short"),
        top_table(r3_long.rename(columns={'tp_atr_s': 'tp_atr_s', 'tp_atr_r': 'tp_atr_r'}), CONFIG).assign(Pass="method_long"),
        top_table(r3_short.rename(columns={'tp_atr_s': 'tp_atr_s', 'tp_atr_r': 'tp_atr_r'}), CONFIG).assign(Pass="method_short"),
    ])
    combined.to_csv("results_atr_sweep.csv", index=False)
    print("\nРезультаты сохранены: results_atr_sweep.csv")
except Exception as e:
    print("\nНе удалось сохранить CSV:", e)

Лучший метод для ЛОНГ : Pivot (классика)
  Буфер S : 0.15 x ATR | SL S: 1.50 x ATR | TP S: 4.00 x ATR
Лучший метод для ШОРТ : DeMark
  Буфер R : 0.10 x ATR | SL R: 1.50 x ATR | TP R: 2.50 x ATR

Правило постановки ордеров (уровни пересчитываются каждый день):
  Buy  Limit = S + 0.15 * ATR   (лонг от поддержки S, метод Pivot (классика))
  Sell Limit = R - 0.10 * ATR   (шорт от сопротивления R, метод DeMark)

Пример на последнем баре (2025-12-30 21:00:00):
  ATR(14) = 0.00108, Close = 1.17466
  S = 1.17306 (piv_s2)  -> Buy  Limit = 1.17322
  R = 1.18004 (dem_r1)  -> Sell Limit = 1.17993

Результаты сохранены: results_atr_sweep.csv


## Как читать результат

1. **Проход 1** — лучшие множители лонга (`buffer_atr_s`, `sl_atr_s`, `tp_atr_s`) и метод.
2. **Проход 2** — лучшие множители шорта (`buffer_atr_r`, `sl_atr_r`, `tp_atr_r`) и метод.
3. **Проход 3a** — выбор лучшего метода для ЛОНГ при лучших S-множителях (`long_only=True`).
4. **Проход 3b** — выбор лучшего метода для ШОРТ при лучших R-множителях (`short_only=True`).
5. **Проверка на 2025** — результаты лучших комбинаций лонг и шорт отдельно на данных, не участвовавших в подборе.

⚠️ Напоминание: комиссия ~0.002%, спред не моделируется; граница дня по умолчанию —
полночь UTC (если в MT5 серверные сутки, задайте `day_offset_hours` = 2 или 3).

In [17]:
# Повторный вывод итогов с русскими заголовками (синхронизировано с подбором)
print('\n--- Обновлённые лучшие методы (Breakout/Bounce) для LONG и SHORT ---')
print('ПРОБОЙ СОПРОТИВЛЕНИЯ — ПРОХОД: ПРОБОЙ СОПРОТИВЛЕНИЯ (BUY STOP) :', LEVEL_METHOD_NAMES[int(brk_best_long['level_method'])], ' — params S:', {
    'buffer_atr_s': float(brk_best1['buffer_atr_s']), 'sl_atr_s': float(brk_best1['sl_atr_s']), 'tp_atr_s': float(brk_best1.get('tp_atr_s', 2.0))
})
print('ПРОБОЙ ПОДДЕРЖКИ — ПРОХОД: ПРОБОЙ ПОДДЕРЖКИ (SELL STOP) :', LEVEL_METHOD_NAMES[int(brk_best_short['level_method'])], ' — params R:', {
    'buffer_atr_r': float(brk_best2['buffer_atr_r']), 'sl_atr_r': float(brk_best2['sl_atr_r']), 'tp_atr_r': float(brk_best2.get('tp_atr_r', 2.0))
})
print('ОТСКОК ОТ ПОДДЕРЖКИ — ПРОХОД: ОТСКОК ОТ ПОДДЕРЖКИ (Buy Limit) :', LEVEL_METHOD_NAMES[int(best_long['level_method'])], ' — params S:', {
    'buffer_atr_s': float(best_long['buffer_atr_s']), 'sl_atr_s': float(best_long['sl_atr_s']), 'tp_atr_s': float(best_long.get('tp_atr_s', 2.0))
})
print('ОТСКОК ОТ СОПРОТИВЛЕНИЯ — ПРОХОД: ОТСКОК ОТ СОПРОТИВЛЕНИЯ (Sell Limit) :', LEVEL_METHOD_NAMES[int(best_short['level_method'])], ' — params R:', {
    'buffer_atr_r': float(best_short['buffer_atr_r']), 'sl_atr_r': float(best_short['sl_atr_r']), 'tp_atr_r': float(best_short.get('tp_atr_r', 2.0))
})


--- Обновлённые лучшие методы (Breakout/Bounce) для LONG и SHORT ---
ПРОБОЙ СОПРОТИВЛЕНИЯ — ПРОХОД: ПРОБОЙ СОПРОТИВЛЕНИЯ (BUY STOP) : Camarilla  — params S: {'buffer_atr_s': 0.25, 'sl_atr_s': 2.0, 'tp_atr_s': 2.0}
ПРОБОЙ ПОДДЕРЖКИ — ПРОХОД: ПРОБОЙ ПОДДЕРЖКИ (SELL STOP) : Pivot (классика)  — params R: {'buffer_atr_r': 0.25, 'sl_atr_r': 1.0, 'tp_atr_r': 2.5}
ОТСКОК ОТ ПОДДЕРЖКИ — ПРОХОД: ОТСКОК ОТ ПОДДЕРЖКИ (Buy Limit) : Pivot (классика)  — params S: {'buffer_atr_s': 0.15, 'sl_atr_s': 1.5, 'tp_atr_s': 4.0}
ОТСКОК ОТ СОПРОТИВЛЕНИЯ — ПРОХОД: ОТСКОК ОТ СОПРОТИВЛЕНИЯ (Sell Limit) : DeMark  — params R: {'buffer_atr_r': 0.1, 'sl_atr_r': 1.5, 'tp_atr_r': 2.5}
